# Fed-KAN week 2: FedAvg vs traffic-aware aggregation

Compares FedAvg with the traffic-aware custom aggregation of Kaur, Sehra et al. (MLWA 2026) over 4 LargeST districts, 15-minute data, 5 rounds, seeds 0, 1 and 2. Heads: Taylor KAN order 1 and the FC baseline, both 64 wide. Three aggregation settings:

- `fedavg`: weights by training examples
- `traffic`: Eq. 14 as printed, so a higher sigma and Psi raise a client's weight
- `traffic_penalize`: sigma and Psi scored as (1 - normalised value), so unstable clients get less weight, as the paper's text describes

The six aggregation coefficients are chosen first by a small random search scored on the **validation** split (8 sets in [0.1, 1.5], seed 0, 3 rounds, Taylor head, `traffic`). Both traffic settings then use the best set. The test split is used only for the final runs, never to pick coefficients. Results are reported overall and per district.

Runtime > Change runtime type > T4 GPU. Expects `MyDrive/fedkan/fedkan_code_v4.zip` and `MyDrive/fedkan/data_real/district_*`. Results go to `MyDrive/fedkan/results_week2`; finished runs are skipped on a rerun.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ROUNDS = 5
RESAMPLE = 3
BATCH = 64
SEEDS = [0, 1, 2]
FLOOR = 0.1          # final weight = (1 - FLOOR) * traffic + FLOOR * fedavg
HEADS = [
    ('taylor_o1_h64', '--head taylor --taylor-order 1 --kan-hidden 64'),
    ('fc_h64',        '--head fc --kan-hidden 64'),
]
AGGS = ['fedavg', 'traffic', 'traffic_penalize']
AGG_FLAGS = {
    'fedavg':           '--agg fedavg',
    'traffic':          '--agg traffic',                              # Eq. 14 as printed
    'traffic_penalize': '--agg traffic --agg-penalize-instability',   # sigma, Psi scored as 1 - value
}

# coefficient search, validation split only
SEARCH_N = 8
SEARCH_ROUNDS = 3
SEARCH_SEED = 0
SEARCH_HEAD = 'taylor_o1_h64'

DRIVE = '/content/drive/MyDrive/fedkan'
WORK = '/content/fedkan'
OUT = f'{DRIVE}/results_week2'
SEARCH_OUT = f'{OUT}/coef_search'

In [ ]:
# copy code and data off Drive onto the local disk
import os, shutil, zipfile
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
with zipfile.ZipFile(f'{DRIVE}/fedkan_code_v4.zip') as z:
    z.extractall(WORK)
shutil.copytree(f'{DRIVE}/data_real', f'{WORK}/phase2/data_real')
os.makedirs(SEARCH_OUT, exist_ok=True)
print(sorted(os.listdir(f'{WORK}/phase2/data_real')))

In [ ]:
%cd /content/fedkan/phase2
BASE = f'--data data_real --resample {RESAMPLE} --batch-size {BATCH}'

In [ ]:
# unit tests for the aggregation weights, then a smoke test of both agg modes (a few batches each)
!python -m unittest discover -s tests
for agg in AGGS:
    agg_flags = AGG_FLAGS[agg]
    !python run_federated_local.py --head taylor --taylor-order 1 {BASE} --rounds 1 --max-batches 3 --max-val-batches 3 --eval-split test {agg_flags}

In [ ]:
# coefficient search on the VALIDATION split: 8 random (alpha, beta, gamma, delta, epsilon, eta) in [0.1, 1.5]
import json
import numpy as np
rng = np.random.default_rng(0)
COEF_SETS = [','.join(f'{c:.3f}' for c in rng.uniform(0.1, 1.5, 6)) for _ in range(SEARCH_N)]
search_flags = dict(HEADS)[SEARCH_HEAD]
for i, coef in enumerate(COEF_SETS):
    name = f'c{i}'
    if os.path.exists(f'{SEARCH_OUT}/{name}.json'):
        print('skip', name, coef)
        continue
    !python run_federated_local.py {search_flags} {BASE} --rounds {SEARCH_ROUNDS} --seed {SEARCH_SEED} --eval-split val --agg traffic --agg-coef {coef} --agg-floor {FLOOR} --out results/search_{name}.json
    if os.path.exists(f'results/search_{name}.json'):
        shutil.copy(f'results/search_{name}.json', f'{SEARCH_OUT}/{name}.json')

In [ ]:
# pick the coefficients with the lowest validation MAE (test is never looked at here)
search = []
for i, coef in enumerate(COEF_SETS):
    with open(f'{SEARCH_OUT}/c{i}.json') as f:
        r = json.load(f)
    assert r['final']['split'] == 'val', f'c{i} was not scored on val'
    assert np.allclose(r['args']['agg_coef'], [float(x) for x in coef.split(',')]), f'c{i} coef mismatch'
    search.append((r['final']['fp32']['mae'], i, coef, r['final']['fp32']))
search.sort()
BEST_COEF = search[0][2]

search_lines = [f'Coefficient search: {SEARCH_HEAD}, seed {SEARCH_SEED}, {SEARCH_ROUNDS} rounds, floor {FLOOR}, scored on val',
                f"{'set':5}{'alpha,beta,gamma,delta,epsilon,eta':38}{'val MAE':>9}{'RMSE':>8}{'MAPE%':>8}{'R2':>7}"]
for mae, i, coef, m in search:
    search_lines.append(f"{'c' + str(i):5}{coef:38}{mae:>9.3f}{m['rmse']:>8.3f}{m['mape']:>8.2f}{m['r2']:>7.3f}")
search_lines.append(f'best: c{search[0][1]} = {BEST_COEF}')
print('\n'.join(search_lines))
with open(f'{OUT}/best_coef.json', 'w') as f:
    json.dump({'coef': BEST_COEF, 'set': f'c{search[0][1]}', 'val_mae': search[0][0], 'floor': FLOOR}, f, indent=2)

In [ ]:
# test runs: every head x agg at seed 0 first, then seeds 1 and 2; each result goes to Drive as soon as it finishes
for seed in SEEDS:
    for head, flags in HEADS:
        for agg in AGGS:
            name = f'{head}_{agg}_s{seed}'
            if os.path.exists(f'{OUT}/{name}.json'):
                print('skip', name)
                continue
            extra = AGG_FLAGS[agg]
            if agg != 'fedavg':
                extra += f' --agg-coef {BEST_COEF} --agg-floor {FLOOR}'
            !python run_federated_local.py {flags} {BASE} --rounds {ROUNDS} --seed {seed} --eval-split test {extra} --out results/{name}.json
            if os.path.exists(f'results/{name}.json'):
                shutil.copy(f'results/{name}.json', f'{OUT}/{name}.json')

In [ ]:
# summary: mean +- std over seeds (sample std), per-step MAE and average per-client weights, saved to Drive
def ms(xs):
    xs = np.asarray(xs, dtype=float)
    sd = xs.std(ddof=1) if len(xs) > 1 else 0.0
    return f'{xs.mean():.2f} ± {sd:.2f}'

runs = {}
for head, _ in HEADS:
    for agg in AGGS:
        rs = []
        for seed in SEEDS:
            path = f'{OUT}/{head}_{agg}_s{seed}.json'
            if os.path.exists(path):
                with open(path) as f:
                    rs.append(json.load(f))
        if rs:
            runs[(head, agg)] = rs

for (head, agg), rs in runs.items():
    for r in rs:
        assert r['final']['split'] == 'test'
        assert r['agg']['mode'] == ('fedavg' if agg == 'fedavg' else 'traffic')
        assert r['agg'].get('penalize_instability', False) == (agg == 'traffic_penalize')
        if agg != 'fedavg' and not np.allclose(r['args']['agg_coef'], [float(x) for x in BEST_COEF.split(',')]):
            print(f'WARNING: {head} {agg} seed {r["args"]["seed"]} used coef {r["args"]["agg_coef"]}, not {BEST_COEF}')

lines = ['Week 2: FedAvg vs traffic-aware aggregation (Kaur et al. 2026)',
         f'test split, fp32, {ROUNDS} rounds, mean ± sample std over seeds {SEEDS}',
         f'traffic coef (alpha,beta,gamma,delta,epsilon,eta) = {BEST_COEF}, floor = {FLOOR}, picked on val MAE',
         '',
         f"{'run':32}{'seeds':>6}{'MAE':>16}{'RMSE':>16}{'MAPE%':>16}{'R2':>16}"]
for (head, agg), rs in runs.items():
    fin = [r['final']['fp32'] for r in rs]
    lines.append(f"{head + ' ' + agg:32}{len(rs):>6}" + ''.join(f'{ms([m[k] for m in fin]):>16}' for k in ('mae', 'rmse', 'mape', 'r2')))

lines += ['', 'MAE by forecast step (test split)',
          f"{'run':32}" + ''.join(f"{str(15 * (h + 1)) + ' min':>16}" for h in range(5))]
for (head, agg), rs in runs.items():
    lines.append(f"{head + ' ' + agg:32}" + ''.join(f"{ms([r['final']['fp32']['steps'][h]['mae'] for r in rs]):>16}" for h in range(5)))

clients = next(iter(runs.values()))[0]['clients'] if runs else []
for key, label in (('mae', 'MAE'), ('rmse', 'RMSE'), ('mape', 'MAPE%'), ('r2', 'R2')):
    lines += ['', f'{label} by district (test split)',
              f"{'run':32}" + ''.join(f'{c:>16}' for c in clients)]
    for (head, agg), rs in runs.items():
        per = [{c['name']: c for c in r['final']['fp32']['per_client']} for r in rs]
        lines.append(f"{head + ' ' + agg:32}" + ''.join(f'{ms([p[c][key] for p in per]):>16}' for c in clients))

lines += ['', 'Average per-client aggregation weight (over rounds and seeds)',
          f"{'run':32}" + ''.join(f'{c:>16}' for c in clients)]
for (head, agg), rs in runs.items():
    w = np.mean([rd['agg_weights'] for r in rs for rd in r['rounds']], axis=0)
    lines.append(f"{head + ' ' + agg:32}" + ''.join(f'{x:>16.3f}' for x in w))
traffic_runs = {a: rs[0] for (h, a), rs in runs.items() if a != 'fedavg'}
for a, r in traffic_runs.items():
    lines.append(f"{a + ' before floor':32}" + ''.join(f'{x:>16.3f}' for x in r['agg']['traffic_weights']))
if traffic_runs:
    a = next(iter(traffic_runs.values()))['agg']
    lines += ['', 'Client metrics (training split; Phi, Psi = mean |change|, std of change in flow)',
              f"{'client':32}" + ''.join(f'{m:>12}' for m in a['metric_names'])]
    for c, row in zip(clients, a['metrics']):
        lines.append(f'{c:32}' + ''.join(f'{v:>12.3f}' for v in row))

lines += [''] + search_lines
table = '\n'.join(lines)
print(table)
with open(f'{OUT}/summary.txt', 'w', encoding='utf-8') as f:
    f.write(table + '\n')